In [2]:
import numpy as np
import pandas as pd 

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))



/kaggle/input/datasets/mervebzn/file-3/File 3.csv


In [3]:
df= pd.read_csv('/kaggle/input/datasets/mervebzn/file-3/File 3.csv')

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Survived     418 non-null    int64  
 2   Pclass       418 non-null    int64  
 3   Name         418 non-null    object 
 4   Sex          418 non-null    object 
 5   Age          332 non-null    float64
 6   SibSp        418 non-null    int64  
 7   Parch        418 non-null    int64  
 8   Ticket       418 non-null    object 
 9   Fare         417 non-null    float64
 10  Cabin        91 non-null     object 
 11  Embarked     418 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 39.3+ KB


In [4]:
df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,418.000000,418.000000,418.000000,332.000000,418.000000,418.000000,417.000000
mean,1100.500000,0.363636,2.265550,30.272590,0.447368,0.392344,35.627188
std,120.810458,0.481622,0.841838,14.181209,0.896760,0.981429,55.907576
min,892.000000,0.000000,1.000000,0.170000,0.000000,0.000000,0.000000
25%,996.250000,0.000000,1.000000,21.000000,0.000000,0.000000,7.895800
50%,1100.500000,0.000000,3.000000,27.000000,0.000000,0.000000,14.454200
75%,1204.750000,1.000000,3.000000,39.000000,1.000000,0.000000,31.500000
max,1309.000000,1.000000,3.000000,76.000000,8.000000,9.000000,512.329200


In [5]:
df["Sex"].value_counts()


Sex
male      266
female    152
Name: count, dtype: int64

In [6]:
df["Pclass"].value_counts()

Pclass
3    218
1    107
2     93
Name: count, dtype: int64

In [7]:
df["Embarked"].value_counts()

Embarked
S    270
C    102
Q     46
Name: count, dtype: int64

# Data Quality


In [8]:
df.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64

In [9]:
# Missing Fare
missing_fare_row= df[df['Fare'].isna()]
print('Missing fare row:\n',missing_fare_row)

Missing fare row:
      PassengerId  Survived  Pclass                Name   Sex   Age  SibSp  \
152         1044         0       3  Storey, Mr. Thomas  male  60.5      0   

     Parch Ticket  Fare Cabin Embarked  
152      0   3701   NaN   NaN        S  


In [10]:
median_fare_pclass3= df[df['Pclass']== 3]['Fare'].median()
df['Fare'] = df['Fare'].fillna(median_fare_pclass3)
df['Fare'].isna().sum()

np.int64(0)

In [11]:
# Missing Age
df[df["Age"].isna()][["Age", "Pclass", "Sex", "Fare", "Survived"]]

,Age,Pclass,Sex,Fare,Survived
10,NaN,3,male,7.8958,0
22,NaN,1,female,31.6833,1
29,NaN,3,male,21.6792,0
33,NaN,3,female,23.4500,1
36,NaN,3,female,8.0500,1
...,...,...,...,...,...
408,NaN,3,female,7.7208,1
410,NaN,3,female,7.7500,1
413,NaN,3,male,8.0500,0
416,NaN,3,male,8.0500,0


In [12]:
df['Age'] =df.groupby(['Pclass', 'Sex'])['Age'].transform(lambda x: x.fillna(x.median()))
df['Age'].isna().sum()

np.int64(0)

In [13]:
# Missing Cabin
df['Has_Cabin']= df['Cabin'].notna().astype(int)

In [14]:
print(df.isna().sum())

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age              0
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          327
Embarked         0
Has_Cabin        0
dtype: int64


In [15]:
# Cabin: extrac usable signal (Has_Cabin, Deck), then drop the raw column
df["Deck"]= df['Cabin'].str[0].fillna('Unknow')
df= df.drop(columns=['Cabin'])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Survived     418 non-null    int64  
 2   Pclass       418 non-null    int64  
 3   Name         418 non-null    object 
 4   Sex          418 non-null    object 
 5   Age          418 non-null    float64
 6   SibSp        418 non-null    int64  
 7   Parch        418 non-null    int64  
 8   Ticket       418 non-null    object 
 9   Fare         418 non-null    float64
 10  Embarked     418 non-null    object 
 11  Has_Cabin    418 non-null    int64  
 12  Deck         418 non-null    object 
dtypes: float64(2), int64(6), object(5)
memory usage: 42.6+ KB


In [16]:
# Duplicate Row Check 
print('Duplicate row count: ', df.duplicated().sum())

Duplicate row count:  0


In [17]:
# Outlier Check
numeric_cols = ['Age','Fare', 'SibSp', 'Parch']

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[
    (df[col]< lower_bound) |
    (df[col]> upper_bound)
    ]
    
    print(f'{col}: {len(outliers)} outliers')
    print(f'Lower Bound: {lower_bound:.2f}')
    print(f'Upper Bound: {upper_bound:.2f}')
    print('-'* 40)


Age: 18 outliers
Lower Bound: 0.44
Upper Bound: 57.94
----------------------------------------
Fare: 55 outliers
Lower Bound: -27.47
Upper Bound: 66.84
----------------------------------------
SibSp: 11 outliers
Lower Bound: -1.50
Upper Bound: 2.50
----------------------------------------
Parch: 94 outliers
Lower Bound: 0.00
Upper Bound: 0.00
----------------------------------------


Age, Fare, SibSp, and Parch were checked using the IQR method. No rows were removed: the extreme values in Age represent real infants/elderly passengers, in Fare represent real 1st-class fares, and in SibSp/Parch represent real large families. Also, the IQR method isn't suitable technique for skewed count like SibSp/Parch in the first place.

In [18]:
print(pd.crosstab(
    df["Sex"],
    df["Survived"],
    margins=True
))

Survived    0    1  All
Sex                    
female      0  152  152
male      266    0  266
All       266  152  418


# Feature Engineering

In [19]:
# Family Size

df['FamilySize'] = df['SibSp'] + df['Parch'] +1

df['IsAlone'] = (df['FamilySize']==1).astype(int)

print(df[['SibSp','Parch','FamilySize','IsAlone']].head())
print("Alone people: ", df['IsAlone'].sum())

   SibSp  Parch  FamilySize  IsAlone
0      0      0           1        1
1      1      0           2        0
2      0      0           1        1
3      0      0           1        1
4      1      1           3        0
Alone people:  253


In [20]:
# Age Groups
bins= [0,12,18,60,100]
labels= ['Child','Teen', 'Adult', 'Senior']
df['AgeGroup']= pd.cut(df['Age'], bins = bins, labels = labels)

print(df['AgeGroup'].value_counts())

AgeGroup
Adult     353
Teen       29
Child      25
Senior     11
Name: count, dtype: int64


In [21]:
# Data Type Check
categorical_cols = ['Sex', 'Embarked', 'Deck']
for col in categorical_cols:
    df[col]= df[col].astype('category')

print(df.dtypes)
print('-' * 40)
print(df.isna().sum())

PassengerId       int64
Survived          int64
Pclass            int64
Name             object
Sex            category
Age             float64
SibSp             int64
Parch             int64
Ticket           object
Fare            float64
Embarked       category
Has_Cabin         int64
Deck           category
FamilySize        int64
IsAlone           int64
AgeGroup       category
dtype: object
----------------------------------------
PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
Has_Cabin      0
Deck           0
FamilySize     0
IsAlone        0
AgeGroup       0
dtype: int64


In [23]:
df.to_csv('cleaned_titanic.csv', index=False)